# M3T2 - Productor de eventos de pedidos

Este cuaderno genera eventos JSON de pedidos para simular una fuente continua de datos. El consumidor de la práctica leerá estos ficheros con Spark Structured Streaming y los escribirá en distintos sinks.

Ejecuta este cuaderno en paralelo con `M3T2_consumidor_sinks_spark.ipynb` si quieres observar el flujo completo.

## 1. Preparación de carpetas

Los eventos se escribirán en `input/pedidos_events`. Cada fichero contiene varios eventos en formato JSON Lines.

In [ ]:
import os
import json
import time
import random
from datetime import datetime, timezone

INPUT_DIR = os.path.abspath("input/pedidos_events")
os.makedirs(INPUT_DIR, exist_ok=True)

print("Directorio de salida:", INPUT_DIR)

## 2. Configuración del productor

Puedes cambiar el número de iteraciones, el tiempo entre lotes y los catálogos de clientes, países o estados para generar más variedad de eventos.

In [ ]:
CLIENTES = ["cli-001", "cli-002", "cli-003", "cli-004", "cli-005"]
PAISES = ["ES", "FR", "PT", "DE", "IT"]
ESTADOS = ["creado", "pagado", "preparado", "enviado", "cancelado"]
CANALES = ["web", "mobile", "marketplace"]

EVENTS_PER_BATCH = 8
SLEEP_SECONDS = 5
MAX_ITERATIONS = 20  # Cambia a None para ejecución indefinida

## 3. Función de generación de eventos

Cada evento incluye un identificador único, una clave de particionado (`pedido_id`), tiempo de evento, payload de negocio y metadatos básicos.

In [ ]:
def crear_evento(iteracion, posicion):
    now = datetime.now(timezone.utc)
    pedido_id = f"ped-{random.randint(1000, 9999)}"
    estado = random.choice(ESTADOS)
    importe = round(random.uniform(15, 450), 2)
    unidades = random.randint(1, 6)

    return {
        "event_id": f"evt-{int(time.time())}-{iteracion}-{posicion}",
        "event_type": f"pedido_{estado}",
        "event_time": now.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "source": "simulador_pedidos",
        "pedido_id": pedido_id,
        "cliente_id": random.choice(CLIENTES),
        "pais": random.choice(PAISES),
        "canal": random.choice(CANALES),
        "estado": estado,
        "importe": importe,
        "unidades": unidades,
        "metadata": {
            "schema_version": "1.0",
            "producer": "M3T2_productor_eventos_pedidos"
        }
    }

crear_evento(1, 1)

## 4. Generación continua de lotes

Cada iteración escribe un fichero nuevo. Spark Structured Streaming detectará esos ficheros como nuevos microbatches.

In [ ]:
total_generados = 0
iterations = range(MAX_ITERATIONS if MAX_ITERATIONS is not None else 10**9)

for i in iterations:
    filename = os.path.join(INPUT_DIR, f"pedidos_{int(time.time())}_{i}.json")
    eventos = [crear_evento(i + 1, j + 1) for j in range(EVENTS_PER_BATCH)]

    with open(filename, "w", encoding="utf-8") as file:
        for event in eventos:
            file.write(json.dumps(event, ensure_ascii=False) + "\n")

    total_generados += len(eventos)
    print(f"[{i + 1}] {len(eventos)} eventos escritos en {filename}. Total: {total_generados}")
    time.sleep(SLEEP_SECONDS)